# Retail Sales Analysis
# Notebook 6 — Streamlit Preparation



---
## 1. Импорт и загрузка

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

print('Библиотеки загружены')

Библиотеки загружены


In [ ]:
customers_features = pd.read_csv('customer_segments.csv')
cluster_summary    = pd.read_csv('cluster_summary.csv', index_col=0)

print(f'customers_features: {customers_features.shape}')
print(f'cluster_summary:    {cluster_summary.shape}')
display(customers_features.head())

customers_features: (98759, 10)
cluster_summary:    (4, 7)


,customer_id,total_spent,avg_check,num_checks,total_items,avg_items_per_check,avg_unique_products,recency,cluster,segment_name
0,1,3135.08003,49.763175,63,63,1.0,1.0,4,2,Low value customers
1,2,3355.04483,53.254680,63,63,1.0,1.0,6,2,Low value customers
2,3,3318.65282,47.409326,70,70,1.0,1.0,1,2,Low value customers
3,4,3122.56073,45.254503,69,69,1.0,1.0,2,2,Low value customers
4,5,2650.34531,44.921107,59,59,1.0,1.0,1,2,Low value customers


In [ ]:
FEATURE_COLS = ['total_spent', 'avg_check', 'num_checks',
                'total_items', 'avg_items_per_check', 'recency']

features = customers_features[FEATURE_COLS].fillna(0)

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

N_CLUSTERS = len(cluster_summary)
print(f'Число кластеров: {N_CLUSTERS}')

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans.fit(features_scaled)

print('Модель обучена')

Число кластеров: 4
Модель обучена


---
## 3. Сохранение модели и скейлера

In [4]:
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Сохранены: kmeans_model.pkl, scaler.pkl')

Сохранены: kmeans_model.pkl, scaler.pkl


---
## 4. Сохранение таблиц для Streamlit

In [ ]:
cluster_names = (
    customers_features[['cluster', 'segment_name']]
    .drop_duplicates()
    .sort_values('cluster')
    .reset_index(drop=True)
)
cluster_names.to_csv('cluster_names.csv', index=False)

cluster_summary_named = cluster_summary.copy()
cluster_summary_named['segment_name'] = cluster_summary_named.index.map(
    cluster_names.set_index('cluster')['segment_name']
)
cluster_summary_named.to_csv('cluster_summary.csv')

print('Сохранены:')
print('  cluster_summary.csv  — профили кластеров с именами')
display(cluster_summary_named)

Сохранены:
  cluster_names.csv    — маппинг cluster → segment_name
  cluster_summary.csv  — профили кластеров с именами


,total_spent,avg_check,num_checks,total_items,avg_items_per_check,recency,size,segment_name
cluster,,,,,,,,
0,43553.52,652.33,67.36,887.65,13.30,1.84,29631,Regular customers
1,42904.38,681.27,63.33,872.36,13.85,6.86,7993,At-risk customers
2,15301.17,228.10,67.44,311.79,4.65,2.21,31576,Low value customers
3,73501.24,1059.59,69.66,1484.62,21.40,2.12,29559,VIP customers


---
## 5. Проверка: predict на тестовом клиенте

In [ ]:
with open('kmeans_model.pkl', 'rb') as f:
    km_loaded = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    sc_loaded = pickle.load(f)

# Тестовый клиент

test_client = pd.DataFrame([{
    'total_spent': 80000,
    'avg_check': 1100,
    'num_checks': 75,
    'total_items': 1600,
    'avg_items_per_check': 21,
    'recency': 2
}])



test_scaled  = sc_loaded.transform(test_client)
pred_cluster = km_loaded.predict(test_scaled)[0]
pred_segment = cluster_names.set_index('cluster').loc[pred_cluster, 'segment_name']

print(f'Тестовый клиент - кластер {pred_cluster}: "{pred_segment}"')

Тестовый клиент -> кластер 3: "VIP customers"
